# 05 - Postulate 4: Composition

**The state space of a composite quantum system is the tensor product of the state spaces of the component systems.**

In plain English: when you combine two qubits, you use the tensor product. We already covered tensor products in the linear algebra section. This postulate is what makes that math physical.

The key consequence: **entanglement**. Some multi-qubit states cannot be described by describing each qubit separately. The whole is more than the sum of its parts.

In [ ]:
import numpy as np

## Product States (Separable)

If qubit A is in state |a> and qubit B is in state |b>, and they are independent, the combined state is:

`|system> = |a> tensor |b>`

This is called a product state. It means each qubit can be described on its own.

In [ ]:
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)

# Product state: qubit A in |+>, qubit B in |0>
product_state = np.kron(ket_plus, ket_0)
print("Product state |+0>:")
print(f"  {product_state.round(4)}")

# Each qubit has its own definite state
# Measuring qubit A tells you nothing about qubit B
print("\nMeasurement probabilities:")
for i, label in enumerate(['00', '01', '10', '11']):
    print(f"  P(|{label}>) = {abs(product_state[i])**2:.4f}")

print("\nQubit A: 50/50. Qubit B: always 0.")
print("They are independent -- no correlation.")

## Entangled States

Some 2-qubit states cannot be written as a tensor product of single-qubit states. These are entangled states.

In an entangled state, the qubits are correlated in a way that has no classical explanation. Measuring one qubit instantly determines the state of the other, no matter how far apart they are.

In [ ]:
# Bell state: (|00> + |11>) / sqrt(2)
bell = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)
print("Bell state |B00>:")
print(f"  {bell.round(4)}")

print("\nMeasurement probabilities:")
for i, label in enumerate(['00', '01', '10', '11']):
    print(f"  P(|{label}>) = {abs(bell[i])**2:.4f}")

print("\n50% |00>, 50% |11>. Never |01> or |10>.")
print("If qubit A is 0, qubit B MUST be 0.")
print("If qubit A is 1, qubit B MUST be 1.")
print("This correlation exists even if the qubits are light-years apart.")

## The Four Bell States

There are four maximally entangled 2-qubit states, called Bell states. They form a complete basis for the 2-qubit system.

In [ ]:
# All four Bell states
phi_plus  = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)
phi_minus = (np.kron(ket_0, ket_0) - np.kron(ket_1, ket_1)) / np.sqrt(2)
psi_plus  = (np.kron(ket_0, ket_1) + np.kron(ket_1, ket_0)) / np.sqrt(2)
psi_minus = (np.kron(ket_0, ket_1) - np.kron(ket_1, ket_0)) / np.sqrt(2)

bell_states = {
    "|B00> (Phi+)": phi_plus,
    "|B01> (Phi-)": phi_minus,
    "|B10> (Psi+)": psi_plus,
    "|B11> (Psi-)": psi_minus,
}

for name, state in bell_states.items():
    probs = [abs(state[i])**2 for i in range(4)]
    labels = ['00', '01', '10', '11']
    nonzero = [(l, p) for l, p in zip(labels, probs) if p > 0.001]
    desc = ", ".join([f"|{l}> ({p:.0%})" for l, p in nonzero])
    print(f"{name}: {desc}")

# Verify they are orthogonal
print("\nOrthogonality check:")
states_list = list(bell_states.values())
names_list = list(bell_states.keys())
for i in range(4):
    for j in range(i+1, 4):
        ip = abs(np.dot(states_list[i].conj(), states_list[j]))
        print(f"  <{names_list[i][:5]}|{names_list[j][:5]}> = {ip:.4f}")

## Partial Measurement

In a multi-qubit system, you can measure just one qubit. The other qubit's state updates based on the result.

For the Bell state (|00> + |11>)/sqrt(2):
- Measure qubit A and get 0 --> qubit B is now in state |0>
- Measure qubit A and get 1 --> qubit B is now in state |1>

In [ ]:
# Bell state
bell = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)

# Projection operators for measuring qubit A only
I = np.eye(2, dtype=complex)

# Project qubit A onto |0>, leave qubit B alone
P_A0 = np.kron(np.outer(ket_0, ket_0), I)  # |0><0| tensor I

# Project qubit A onto |1>, leave qubit B alone  
P_A1 = np.kron(np.outer(ket_1, ket_1), I)  # |1><1| tensor I

# Probability of qubit A being 0
prob_A0 = np.real(bell.conj() @ P_A0 @ bell)
print(f"P(qubit A = 0) = {prob_A0:.4f}")

# State after measuring qubit A = 0
post_A0 = (P_A0 @ bell) / np.sqrt(prob_A0)
print(f"State after qubit A = 0: {post_A0.round(4)}")
print("This is |00>. Qubit B collapsed to |0>.")

# Probability of qubit A being 1
prob_A1 = np.real(bell.conj() @ P_A1 @ bell)
print(f"\nP(qubit A = 1) = {prob_A1:.4f}")

# State after measuring qubit A = 1
post_A1 = (P_A1 @ bell) / np.sqrt(prob_A1)
print(f"State after qubit A = 1: {post_A1.round(4)}")
print("This is |11>. Qubit B collapsed to |1>.")

## Entanglement as a Resource

Entanglement is not just a curiosity. It is a resource that enables:

1. **Quantum teleportation**: Transfer a qubit state using entanglement + classical communication
2. **Superdense coding**: Send 2 classical bits using 1 qubit + entanglement
3. **Quantum key distribution**: Detect eavesdroppers using entanglement
4. **Quantum algorithms**: Speed up computation through entanglement-driven interference

Without entanglement, a quantum computer is no more powerful than a classical one.

## How to Check if a State is Entangled

For a 2-qubit pure state, reshape the state vector into a 2x2 matrix and compute its Schmidt decomposition (or just check the rank). If the rank is 1, it is separable. If the rank is 2, it is entangled.

In [ ]:
def is_entangled(state_2qubit):
    """Check if a 2-qubit state is entangled using Schmidt decomposition."""
    # Reshape into 2x2 matrix
    matrix = state_2qubit.reshape(2, 2)
    # Singular value decomposition
    U, S, Vh = np.linalg.svd(matrix)
    # Count significant singular values
    significant = np.sum(S > 1e-10)
    return significant > 1, S

# Test: product state |+0>
product = np.kron(ket_plus, ket_0)
entangled, sv = is_entangled(product)
print(f"|+0> entangled? {entangled}  Schmidt values: {sv.round(4)}")

# Test: Bell state
bell = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)
entangled, sv = is_entangled(bell)
print(f"Bell  entangled? {entangled}  Schmidt values: {sv.round(4)}")

# Test: partially entangled state
partial = np.sqrt(0.8) * np.kron(ket_0, ket_0) + np.sqrt(0.2) * np.kron(ket_1, ket_1)
entangled, sv = is_entangled(partial)
print(f"Partial entangled? {entangled}  Schmidt values: {sv.round(4)}")

## Exercises

1. Create the state (|01> + |10>)/sqrt(2). Is it entangled? What happens when you measure qubit A?

2. Create a 3-qubit GHZ state: (|000> + |111>)/sqrt(2). Measure qubit A. What happens to qubits B and C?

3. Start with |00>, apply H to qubit A (H tensor I), then apply CNOT. Verify you get the Bell state. This is the standard Bell state preparation circuit.

In [ ]:
# Your code here
